In [ ]:
# import library
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import html
import csv
import time

In [ ]:
def scrape_links(url, session=None, sleep_between_requests=0.5):
    if session is None:
        session = requests.Session()
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; Bot/0.1; +https://example.com/bot)" # user agent untuk membuka peramban scraping
    }
    resp = session.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # cari anchors khusus di blok tombol listing ("Lihat Detail")
    anchors = soup.select("div.listing-card__information__bottom__buttons a")
    # mengambil semua elemen a yang memiliki href mengandung '/adform/'
    if not anchors:
        anchors = [a for a in soup.find_all("a", href=True) if "/adform/" in a["href"]]

    results = [] # membuat list kosong
    seen = set()
    for a in anchors: # looping setiap link
        raw = a.get("href") # pembersihan link
        if not raw:
            continue
        decoded = html.unescape(raw)
        full = urljoin(resp.url, decoded)
        if full in seen: # cek dan hapus nilai duplikat
            continue
        seen.add(full)
        results.append({
            "text": a.get_text(strip=True),
            "href": decoded,
            "full_url": full,
            "rel": a.get("rel"),
            "title": a.get("title")
        })

    # Delay scraping
    time.sleep(sleep_between_requests)
    return results

if __name__ == "__main__":
    url = "https://rumah.mitula.co.id/find?page=40&geoId=mitula-ID-poblacion-0000155438&operationType=sell&propertyType=house" #link scraping
    links = scrape_links(url)

    # print hasil scrap
    for i,l in enumerate(links, start=1):
        print(f"{i}. {l['text']} -> {l['full_url']}")

    # simpan CSV
    keys = ["text","href","full_url","title","rel"]
    with open("data_scrap/mitula_links_page40.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(links)

    print(f"Saved {len(links)} links to mitula_links_page1.csv")


1. Lihat detail -> https://rumah.mitula.co.id/adform/24301-256-4b88-fdd53dce00a6-b965-8ef872b4-f235?page=40&pos=0&t_sec=1&t_pvid=1e8a780a-b818-4080-967a-b2b9e0c97a9a&hd=false
2. Lihat detail -> https://rumah.mitula.co.id/adform/24301-256-4d67-1db614ad839c-b8c0-5bb0184f-1d6d?page=40&pos=1&t_sec=1&t_pvid=1e8a780a-b818-4080-967a-b2b9e0c97a9a&hd=false
3. Lihat detail -> https://rumah.mitula.co.id/adform/24301-256-4f0d-9a0e2ec0ac93-817b-347d0e97-6ea?page=40&pos=2&t_sec=1&t_pvid=1e8a780a-b818-4080-967a-b2b9e0c97a9a&hd=false
4. Lihat detail -> https://rumah.mitula.co.id/adform/24301-256-4c17-28e07316192a-c059-f689880d-fda6?page=40&pos=3&t_sec=1&t_pvid=1e8a780a-b818-4080-967a-b2b9e0c97a9a&hd=false
5. Lihat detail -> https://rumah.mitula.co.id/adform/24301-256-4b33-a8a3725ed327-9ac9-b75c0713-8918?page=40&pos=4&t_sec=1&t_pvid=1e8a780a-b818-4080-967a-b2b9e0c97a9a&hd=false
6. Lihat detail -> https://rumah.mitula.co.id/adform/24301-256-5081-88d79eaa6c76-8b45-17284f56-7988?page=40&pos=5&t_sec=1&t_pv

In [ ]:
import pandas as pd # import library
hai = pd.read_csv("mitula_links_page2.csv") # mengambil data
hai['full_url'] # menampilkan data

,full_url
0,https://rumah.mitula.co.id/adform/24301-256-74...
1,https://rumah.mitula.co.id/adform/24301-256-7b...
2,https://rumah.mitula.co.id/adform/24301-256-79...
3,https://rumah.mitula.co.id/adform/24301-256-7d...
4,https://rumah.mitula.co.id/adform/24301-256-78...
5,https://rumah.mitula.co.id/adform/24301-256-7b...
6,https://rumah.mitula.co.id/adform/24301-256-78...
7,https://rumah.mitula.co.id/adform/24301-256-78...
8,https://rumah.mitula.co.id/adform/24301-256-78...
9,https://rumah.mitula.co.id/adform/24301-256-73...


# Menggabungkan beberapa file menjadi 1 file

In [ ]:
# import library
import pandas as pd
import glob

# mengambil direktori data
all_files = glob.glob('/content/data_scrap/*.csv')

# membuat list kosong untuk menyimpan data yang ingin digabungkan
df_list = []

# membuat file csv dengan loop
for f in all_files:
    df = pd.read_csv(f)
    df_list.append(df)

# mengombinasikan hasil file .csv
combined_df = pd.concat(df_list, ignore_index=True)

# menampilkan dataframe
print("Combined DataFrame shape:", combined_df.shape)
display(combined_df.head())

# menyimpan data menjadi format .csv
combined_df.to_csv("combined_mitula_links.csv", index=False)
print("Combined CSV file saved as combined_mitula_links.csv")

Combined DataFrame shape: (1200, 5)


,text,href,full_url,title,rel
0,Lihat detail,/adform/24301-256-798a-36df367633ff-9a53-19198...,https://rumah.mitula.co.id/adform/24301-256-79...,Lihat detail,['nofollow']
1,Lihat detail,/adform/24301-256-7fef-72fa3c7539ef-a760-191e6...,https://rumah.mitula.co.id/adform/24301-256-7f...,Lihat detail,['nofollow']
2,Lihat detail,/adform/24301-256-7835-c00caa41e46d-b9be-191e5...,https://rumah.mitula.co.id/adform/24301-256-78...,Lihat detail,['nofollow']
3,Lihat detail,/adform/24301-256-7438-c64d5e46a3a1-a4cd-191dc...,https://rumah.mitula.co.id/adform/24301-256-74...,Lihat detail,['nofollow']
4,Lihat detail,/adform/24301-256-74c7-64246b4dcfb6-bb14-191c5...,https://rumah.mitula.co.id/adform/24301-256-74...,Lihat detail,['nofollow']


Combined CSV file saved as combined_mitula_links.csv


In [ ]:
combine = pd.read_csv("combined_mitula_links.csv")
combine

,text,href,full_url,title,rel
0,Lihat detail,/adform/24301-256-798a-36df367633ff-9a53-19198...,https://rumah.mitula.co.id/adform/24301-256-79...,Lihat detail,['nofollow']
1,Lihat detail,/adform/24301-256-7fef-72fa3c7539ef-a760-191e6...,https://rumah.mitula.co.id/adform/24301-256-7f...,Lihat detail,['nofollow']
2,Lihat detail,/adform/24301-256-7835-c00caa41e46d-b9be-191e5...,https://rumah.mitula.co.id/adform/24301-256-78...,Lihat detail,['nofollow']
3,Lihat detail,/adform/24301-256-7438-c64d5e46a3a1-a4cd-191dc...,https://rumah.mitula.co.id/adform/24301-256-74...,Lihat detail,['nofollow']
4,Lihat detail,/adform/24301-256-74c7-64246b4dcfb6-bb14-191c5...,https://rumah.mitula.co.id/adform/24301-256-74...,Lihat detail,['nofollow']
...,...,...,...,...,...
1195,Lihat detail,/adform/24301-256-7b92-dbaf3a37c3bf-8656-19305...,https://rumah.mitula.co.id/adform/24301-256-7b...,Lihat detail,['nofollow']
1196,Lihat detail,/adform/24301-256-7949-652c86138bd3-8107-192f0...,https://rumah.mitula.co.id/adform/24301-256-79...,Lihat detail,['nofollow']
1197,Lihat detail,/adform/24301-256-7cb4-79dc0b8f652c-aebc-194ee...,https://rumah.mitula.co.id/adform/24301-256-7c...,Lihat detail,['nofollow']
1198,Lihat detail,/adform/24301-256-732c-c6b33d14de2f-beab-195d2...,https://rumah.mitula.co.id/adform/24301-256-73...,Lihat detail,['nofollow']


## Debug scraping 1 halaman

In [ ]:
import re
import json
from typing import Optional, Dict, Any, List

import requests
from bs4 import BeautifulSoup

# Optional selenium imports (only used if you set use_selenium=True)
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except Exception:
    SELENIUM_AVAILABLE = False


HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Bot/0.1; +https://example.com/bot)"
}


def _get_soup_requests(url: str, timeout: int = 15) -> Optional[BeautifulSoup]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "lxml")
    except Exception as e:
        # print("requests error:", e)
        return None


def _get_soup_selenium(url: str, timeout: int = 20) -> Optional[BeautifulSoup]:
    if not SELENIUM_AVAILABLE:
        return None
    try:
        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)
        driver.set_page_load_timeout(timeout)
        driver.get(url)
        html = driver.page_source
        driver.quit()
        return BeautifulSoup(html, "lxml")
    except Exception as e:
        try:
            driver.quit()
        except Exception:
            pass
        # print("selenium error:", e)
        return None


def _text_or_none(el) -> Optional[str]:
    if el is None:
        return None
    txt = el.get_text(separator=" ", strip=True)
    return txt if txt != "" else None


def _extract_number_from_text(text: str) -> Optional[int]:
    if not text:
        return None
    m = re.search(r'(\d+)', text.replace(".", ""))
    if m:
        return int(m.group(1))
    return None


def parse_listing(soup: BeautifulSoup) -> Dict[str, Any]:
    result: Dict[str, Any] = {
        "operation_type": None,     # contoh "Beli"
        "price": None,              # contoh, "Rp 550Jt"
        "price_raw": None,          # harga sewa
        "location": None,           # teks view-map
        "bedrooms": None,           # integer atau None
        "bathrooms": None,          # integer atau None
        "guest_toilet": None,       # string or integer
        "place_features": {},       # kamus dari feature_name -> value (strings)
        "facilities_property": [],  # list of strings (Karakteristik properti)
        "facilities_building": [],  # list of strings (Karakteristik bangunan)
        "raw_details_list": []      # baris list of details-item-value strings (for debugging)
    }

    # Harga dan operasi
    price_item = soup.select_one(".prices-and-fees__price-item.selected")
    if price_item:
        op = price_item.select_one(".prices-and-fees__operation-type")
        price = price_item.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)
        result["price_raw"] = price_item.get_text(separator=" ", strip=True)
    else:
        # mencari keseluruhan elemen harga
        op = soup.select_one(".prices-and-fees__operation-type")
        price = soup.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)

    # mengambil Lokasi
    loc = soup.select_one("#view-map__text") or soup.select_one(".view-map__text")
    result["location"] = _text_or_none(loc)

    # detail tempat (kamar tidur, kamar mandi, guest toilet)
    place_details = soup.select_one(".place-details")
    if place_details:
        items = place_details.select(".details-item-value")
        for it in items:
            txt = _text_or_none(it)
            if not txt:
                continue
            result["raw_details_list"].append(txt)

            lower = txt.lower()
            if "kamar tidur" in lower or "bedroom" in lower:
                # ekstrak jumlah kamar tidur
                result["bedrooms"] = _extract_number_from_text(lower)
                if result["bedrooms"] is None:
                    result["bedrooms"] = txt
            elif "kamar mandi" in lower or "bath" in lower:
                # jumlah "kamar mandi"
                if "kamar mandi" in lower:
                    result["bathrooms"] = _extract_number_from_text(lower) or txt
                elif "guest toilet" in lower or "guest" in lower:
                    result["guest_toilet"] = txt
                else:
                    # mengambil jumlah kamar mandi pribadi dan tamu
                    result["bathrooms"] = _extract_number_from_text(lower) or txt
            elif "guest" in lower or "toilet" in lower:
                result["guest_toilet"] = txt
            else:
                pass

    # mengambil fitur (Jenis rumah, Jenis transaksi, Jumlah lantai, Area yang dapat digunakan, Land area, etc.)
    pf = soup.select_one(".place-features")
    if pf:
        for f in pf.select(".feature"):
            b_tag = f.find("b")
            if b_tag:
                key = b_tag.get_text(separator=" ", strip=True).rstrip(":").strip()
                # Remove the bold text from the full div text to get value
                full = f.get_text(separator=" ", strip=True)
                # value is full minus the key text (b_tag text)
                value = full.replace(b_tag.get_text(), "").strip()
                # Normalize whitespace
                value = re.sub(r'\s+', ' ', value).strip()
                result["place_features"][key] = value if value != "" else None
            else:
                # no <b> label, store whole text
                txt = f.get_text(separator=" ", strip=True)
                if txt:
                    result["place_features"].setdefault("other", []).append(txt)

    # mengambil fasilitas rumah
    facilities = soup.select(".facilities")
    for fac in facilities:
        title_el = fac.select_one(".facilities__title")
        title = _text_or_none(title_el)
        # gather all spans inside li's (the actual list of facility labels)
        items: List[str] = []
        # possible structures: ul > div.facilities__item > li > span
        for span in fac.select("li span"):
            t = _text_or_none(span)
            if t:
                items.append(t)
        if not title:
            # if no title, append to generic list
            result.setdefault("facilities_other", []).extend(items)
        else:
            tl = title.lower()
            if "karakteristik properti" in tl or "properti" in tl:
                result["facilities_property"] = items
            elif "karakteristik bangunan" in tl or "bangunan" in tl:
                result["facilities_building"] = items
            else:
                # unknown facilities group
                result.setdefault("facilities_other_named", {})[title] = items

    if not result["raw_details_list"]:
        result["raw_details_list"] = None
    if not result["place_features"]:
        result["place_features"] = None
    if not result["facilities_property"]:
        result["facilities_property"] = None
    if not result["facilities_building"]:
        result["facilities_building"] = None

    return result


def scrape_mitula(url: str, use_selenium_if_needed: bool = True) -> Dict[str, Any]:
    # mencoba permintaan web
    soup = _get_soup_requests(url)
    if soup:
        parsed = parse_listing(soup)
        if parsed.get("price") or parsed.get("location") or parsed.get("raw_details_list"):
            return parsed

    # If not found and selenium is allowed, try selenium
    if use_selenium_if_needed:
        soup2 = _get_soup_selenium(url)
        if soup2:
            return parse_listing(soup2)

    # final fallback: return mostly nulls
    return {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": None,
        "facilities_property": None,
        "facilities_building": None,
        "raw_details_list": None
    }


if __name__ == "__main__":
    # link contoh pemakaian
    url = ("https://rumah.mitula.co.id/adform/24301-256-7b92-dbaf3a37c3bf-8656-193059c-b5a2?page=5&pos=25&t_sec=1&t_pvid=55314a50-e31e-4605-890c-2b3b27096015&hd=false")
    data = scrape_mitula(url, use_selenium_if_needed=True)
    # tampilkan JSON (None -> null)
    print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "operation_type": null,
  "price": "Rp 800Jt",
  "price_raw": "Rp 800Jt",
  "location": "Waru, Jawa Timur, Sidoarjo, Jawa Timur",
  "bedrooms": 3,
  "bathrooms": 2,
  "guest_toilet": "2 Guest toilets",
  "place_features": {
    "Jenis rumah": "Rumah",
    "Jenis transaksi": "Beli",
    "Jumlah lantai": "2",
    "Area yang dapat digunakan": "68 m²",
    "Land area": "72 m²"
  },
  "facilities_property": [
    "Balkon",
    "Air",
    "Tanpa perabotan",
    "Tangki air",
    "Listrik",
    "Teras"
  ],
  "facilities_building": [
    "Keamanan",
    "Cctv"
  ],
  "raw_details_list": [
    "3 kamar tidur",
    "2 kamar mandi",
    "2 Guest toilets"
  ]
}


In [ ]:
# scrape_mitula_batch.py
import re
import json
import time
import random
from typing import Optional, Dict, Any, List
import pandas as pd
from tqdm import tqdm

import requests
from bs4 import BeautifulSoup

# Optional selenium imports (only used if you set use_selenium=True)
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except Exception:
    SELENIUM_AVAILABLE = False

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Bot/0.1; +https://example.com/bot)"
}


def _get_soup_requests(url: str, timeout: int = 15) -> Optional[BeautifulSoup]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "lxml")
    except Exception:
        return None

def _get_soup_selenium(url: str, timeout: int = 20) -> Optional[BeautifulSoup]:
    if not SELENIUM_AVAILABLE:
        return None
    driver = None
    try:
        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)
        driver.set_page_load_timeout(timeout)
        driver.get(url)
        html = driver.page_source
        driver.quit()
        return BeautifulSoup(html, "lxml")
    except Exception:
        try:
            if driver:
                driver.quit()
        except Exception:
            pass
        return None


def _text_or_none(el) -> Optional[str]:
    if el is None:
        return None
    txt = el.get_text(separator=" ", strip=True)
    return txt if txt != "" else None


def _extract_number_from_text(text: str) -> Optional[int]:
    if not text:
        return None
    m = re.search(r'(\d+)', text.replace(".", ""))
    if m:
        try:
            return int(m.group(1))
        except Exception:
            return None
    return None


def parse_listing(soup: BeautifulSoup) -> Dict[str, Any]:
    result: Dict[str, Any] = {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": {},
        "facilities_property": [],
        "facilities_building": [],
        "raw_details_list": []
    }

    # Price & operation
    price_item = soup.select_one(".prices-and-fees__price-item.selected")
    if price_item:
        op = price_item.select_one(".prices-and-fees__operation-type")
        price = price_item.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)
        result["price_raw"] = price_item.get_text(separator=" ", strip=True)
    else:
        op = soup.select_one(".prices-and-fees__operation-type")
        price = soup.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)

    # Location
    loc = soup.select_one("#view-map__text") or soup.select_one(".view-map__text")
    result["location"] = _text_or_none(loc)

    # Place details
    place_details = soup.select_one(".place-details")
    if place_details:
        items = place_details.select(".details-item-value")
        for it in items:
            txt = _text_or_none(it)
            if not txt:
                continue
            result["raw_details_list"].append(txt)
            lower = txt.lower()
            if "kamar tidur" in lower or "bedroom" in lower:
                result["bedrooms"] = _extract_number_from_text(lower) or txt
            elif "kamar mandi" in lower:
                result["bathrooms"] = _extract_number_from_text(lower) or txt
            elif "guest" in lower or "toilet" in lower:
                # guest toilet
                result["guest_toilet"] = txt

    # Place features
    pf = soup.select_one(".place-features")
    if pf:
        for f in pf.select(".feature"):
            b_tag = f.find("b")
            if b_tag:
                key = b_tag.get_text(separator=" ", strip=True).rstrip(":").strip()
                full = f.get_text(separator=" ", strip=True)
                value = full.replace(b_tag.get_text(), "").strip()
                value = re.sub(r'\s+', ' ', value).strip()
                result["place_features"][key] = value if value != "" else None
            else:
                txt = f.get_text(separator=" ", strip=True)
                if txt:
                    result["place_features"].setdefault("other", []).append(txt)

    # Facilities (possibly multiple groups)
    facilities = soup.select(".facilities")
    for fac in facilities:
        title_el = fac.select_one(".facilities__title")
        title = _text_or_none(title_el)
        items: List[str] = []
        for span in fac.select("li span"):
            t = _text_or_none(span)
            if t:
                items.append(t)
        if title and ("properti" in title.lower() or "karakteristik properti" in title.lower()):
            result["facilities_property"].extend(items)
        elif title and ("bangunan" in title.lower() or "karakteristik bangunan" in title.lower()):
            result["facilities_building"].extend(items)
        else:
            # unknown group, put under property if property empty; else append to building
            if not result["facilities_property"]:
                result["facilities_property"].extend(items)
            else:
                result["facilities_building"].extend(items)

    # Normalize empty -> None as requested
    if not result["raw_details_list"]:
        result["raw_details_list"] = None
    if not result["place_features"]:
        result["place_features"] = None
    if not result["facilities_property"]:
        result["facilities_property"] = None
    if not result["facilities_building"]:
        result["facilities_building"] = None

    return result


def scrape_mitula(url: str, use_selenium_if_needed: bool = True) -> Dict[str, Any]:
    soup = _get_soup_requests(url)
    if soup:
        parsed = parse_listing(soup)
        if parsed.get("price") or parsed.get("location") or parsed.get("raw_details_list"):
            return parsed
    if use_selenium_if_needed:
        soup2 = _get_soup_selenium(url)
        if soup2:
            return parse_listing(soup2)
    # fallback all None
    return {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": None,
        "facilities_property": None,
        "facilities_building": None,
        "raw_details_list": None
    }


def main(csv_path: str,
         url_column: str = "full_url",
         output_jsonl: str = "mitula_output.jsonl",
         output_csv: str = "mitula_output.csv",
         use_selenium: bool = False,
         delay_min: float = 1.0,
         delay_max: float = 2.5):
    df = pd.read_csv(csv_path)
    if url_column not in df.columns:
        raise ValueError(f"CSV tidak memiliki kolom '{url_column}'")

    results = []
    with open(output_jsonl, "w", encoding="utf-8") as fout:
        for idx, url in enumerate(tqdm(df[url_column].dropna(), desc="Scraping URLs")):
            try:
                item = scrape_mitula(url, use_selenium_if_needed=use_selenium)
                # include source URL and index
                item_record = {"source_url": url, "index": int(idx), **item}
                results.append(item_record)
                fout.write(json.dumps(item_record, ensure_ascii=False) + "\n")
            except Exception as e:
                # record error row with None values
                err_record = {"source_url": url, "index": int(idx), "error": str(e)}
                results.append(err_record)
                fout.write(json.dumps(err_record, ensure_ascii=False) + "\n")

            # polite crawling delay
            time.sleep(random.uniform(delay_min, delay_max))

    # Also save a flattened CSV (NaN will be used for None)
    df_out = pd.json_normalize(results)
    df_out.to_csv(output_csv, index=False)
    print(f"Saved JSONL -> {output_jsonl}")
    print(f"Saved CSV   -> {output_csv}")


if __name__ == "__main__":
    # contoh pemakaian: ubah path CSV bila perlu
    main(
        csv_path="combined_mitula_links.csv",
        url_column="full_url",
        output_jsonl="mitula_output.jsonl",
        output_csv="mitula_output.csv",
        use_selenium=False,
        delay_min=1.0,
        delay_max=2.0
    )


Scraping URLs: 100%|██████████| 1200/1200 [57:13<00:00,  2.86s/it]


Saved JSONL -> mitula_output.jsonl
Saved CSV   -> mitula_output.csv


In [ ]:
gab = pd.read_csv("combined_mitula_links.csv")
gab.tail(5)

,text,href,full_url,title,rel
1195,Lihat detail,/adform/24301-256-7b92-dbaf3a37c3bf-8656-19305...,https://rumah.mitula.co.id/adform/24301-256-7b...,Lihat detail,['nofollow']
1196,Lihat detail,/adform/24301-256-7949-652c86138bd3-8107-192f0...,https://rumah.mitula.co.id/adform/24301-256-79...,Lihat detail,['nofollow']
1197,Lihat detail,/adform/24301-256-7cb4-79dc0b8f652c-aebc-194ee...,https://rumah.mitula.co.id/adform/24301-256-7c...,Lihat detail,['nofollow']
1198,Lihat detail,/adform/24301-256-732c-c6b33d14de2f-beab-195d2...,https://rumah.mitula.co.id/adform/24301-256-73...,Lihat detail,['nofollow']
1199,Lihat detail,/adform/24301-256-732f-aa92a38d15d0-8821-195cc...,https://rumah.mitula.co.id/adform/24301-256-73...,Lihat detail,['nofollow']


In [ ]:
hasil_scrap = pd.read_csv("mitula_output.csv")
hasil_scrap.tail(5)

,source_url,index,operation_type,price,price_raw,location,bedrooms,bathrooms,guest_toilet,facilities_property,...,place_features.Jenis rumah,place_features.Jenis transaksi,place_features.Tahun konstruksi,place_features.Jumlah lantai,place_features.Area yang dapat digunakan,place_features.Land area,place_features.Jenis kepemilikan,place_features,place_features.Durasi kontrak,place_features.Nama proyek
1195,https://rumah.mitula.co.id/adform/24301-256-7b...,1195,NaN,Rp 800Jt,Rp 800Jt,"Waru, Jawa Timur, Sidoarjo, Jawa Timur",3.0,2.0,2 Guest toilets,"['Balkon', 'Air', 'Tanpa perabotan', 'Tangki a...",...,Rumah,Beli,NaN,2.0,68 m²,72 m²,NaN,NaN,NaN,NaN
1196,https://rumah.mitula.co.id/adform/24301-256-79...,1196,NaN,Rp 665Jt,Rp 665Jt,"Gedangan, Jawa Timur, Sidoarjo, Jawa Timur",2.0,1.0,1 Guest toilet,"['Air', 'Tanpa perabotan', 'Listrik', 'Teras']",...,Rumah,Beli,NaN,1.0,70 m²,90 m²,NaN,NaN,NaN,NaN
1197,https://rumah.mitula.co.id/adform/24301-256-7c...,1197,NaN,Rp 450Jt,Rp 450Jt,"Sidoarjo, Jawa Timur, Sidoarjo, Jawa Timur",2.0,1.0,1 Guest toilet,"['Garasi', 'Ruang layanan', 'Air', 'Tanpa pera...",...,Rumah,Beli,2000.0,1.0,90 m²,90 m²,NaN,NaN,NaN,NaN
1198,https://rumah.mitula.co.id/adform/24301-256-73...,1198,NaN,"Rp 1,20M","Rp 1,20M","Waru, Jawa Timur, Sidoarjo, Jawa Timur",2.0,2.0,NaN,NaN,...,Rumah,Beli,NaN,1.0,135 m²,135 m²,NaN,NaN,NaN,NaN
1199,https://rumah.mitula.co.id/adform/24301-256-73...,1199,NaN,"Rp 1,50M","Rp 1,50M","Buduran, Jawa Timur, Sidoarjo, Jawa Timur",3.0,2.0,2 Guest toilets,"['Garasi', 'AC', 'Sebagian perabotan', 'Intern...",...,Rumah,Beli,NaN,1.0,72 m²,150 m²,NaN,NaN,NaN,NaN


## Hasil integrasi data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv("/content/drive/MyDrive/AI Talent Factory/hasil_scraping.csv")
data.head()

,source_url,index,operation_type,price,price_raw,location,bedrooms,bathrooms,guest_toilet,facilities_property,...,place_features.Jenis rumah,place_features.Jenis transaksi,place_features.Tahun konstruksi,place_features.Jumlah lantai,place_features.Area yang dapat digunakan,place_features.Land area,place_features.Jenis kepemilikan,place_features,place_features.Durasi kontrak,place_features.Nama proyek
0,https://rumah.mitula.co.id/adform/24301-256-79...,0,NaN,Rp 990Jt,Rp 990Jt,"Waru, Jawa Timur, Sidoarjo, Jawa Timur",2.0,1.0,1 Guest toilet,"['Garasi', 'Tangki air', 'Listrik', 'Teras', '...",...,Rumah,Beli,2010.0,1.0,54 m²,135 m²,NaN,NaN,NaN,NaN
1,https://rumah.mitula.co.id/adform/24301-256-7f...,1,NaN,"Rp 1,20M","Rp 1,20M","Gedangan, Jawa Timur, Sidoarjo, Jawa Timur",3.0,2.0,2 Guest toilets,"['Garasi', 'Tanpa perabotan']",...,Rumah,Beli,NaN,2.0,93 m²,105 m²,NaN,NaN,NaN,NaN
2,https://rumah.mitula.co.id/adform/24301-256-78...,2,NaN,Rp 375Jt,Rp 375Jt,"Krembung, Jawa Timur, Sidoarjo, Jawa Timur",2.0,1.0,1 Guest toilet,"['Garasi', 'Pemandangan panorama', 'Ruang laya...",...,Rumah,Beli,NaN,1.0,46 m²,70 m²,Hak milik,NaN,NaN,NaN
3,https://rumah.mitula.co.id/adform/24301-256-74...,3,NaN,Rp 2M,Rp 2M,"Waru, Jawa Timur, Sidoarjo, Jawa Timur",3.0,2.0,NaN,NaN,...,Rumah,Beli,NaN,2.0,105 m²,98 m²,NaN,NaN,NaN,NaN
4,https://rumah.mitula.co.id/adform/24301-256-74...,4,NaN,Rp 720Jt,Rp 720Jt,"Gedangan, Jawa Timur, Sidoarjo, Jawa Timur",2.0,1.0,1 Guest toilet,"['Air', 'Tanpa perabotan', 'Listrik', 'Teras']",...,Rumah,Beli,NaN,1.0,53 m²,90 m²,NaN,NaN,NaN,NaN


## Scrap_New

In [3]:
import re
import json
from typing import Optional, Dict, Any, List

import requests
from bs4 import BeautifulSoup

# Optional selenium imports
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except Exception:
    SELENIUM_AVAILABLE = False


HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Bot/0.1; +https://example.com/bot)"
}


def _get_soup_requests(url: str, timeout: int = 15) -> Optional[BeautifulSoup]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "lxml")
    except Exception as e:
        # print("requests error:", e)
        return None


def _get_soup_selenium(url: str, timeout: int = 20) -> Optional[BeautifulSoup]:
    if not SELENIUM_AVAILABLE:
        return None
    try:
        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)
        driver.set_page_load_timeout(timeout)
        driver.get(url)
        html = driver.page_source
        driver.quit()
        return BeautifulSoup(html, "lxml")
    except Exception as e:
        try:
            driver.quit()
        except Exception:
            pass
        # print("selenium error:", e)
        return None


def _text_or_none(el) -> Optional[str]:
    if el is None:
        return None
    txt = el.get_text(separator=" ", strip=True)
    return txt if txt != "" else None


def _extract_number_from_text(text: str) -> Optional[int]:
    if not text:
        return None
    m = re.search(r'(\d+)', text.replace(".", ""))
    if m:
        return int(m.group(1))
    return None


def parse_listing(soup: BeautifulSoup) -> Dict[str, Any]:
    result: Dict[str, Any] = {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "latitude": None,   # Field Baru
        "longitude": None,  # Field Baru
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": {},
        "facilities_property": [],
        "facilities_building": [],
        "raw_details_list": []
    }

    # --- BAGIAN BARU: EKSTRAKSI LATITUDE & LONGITUDE ---
    # Mengambil seluruh teks HTML (termasuk script JS) sebagai string
    html_content = str(soup)

    # Mencari pola: latitude: "-7.xxx" (dengan handling spasi dan kutip ' atau ")
    lat_match = re.search(r'latitude\s*:\s*["\']([^"\']+)["\']', html_content)
    lon_match = re.search(r'longitude\s*:\s*["\']([^"\']+)["\']', html_content)

    if lat_match:
        result["latitude"] = lat_match.group(1)
    if lon_match:
        result["longitude"] = lon_match.group(1)
    # ---------------------------------------------------

    # Harga dan operasi
    price_item = soup.select_one(".prices-and-fees__price-item.selected")
    if price_item:
        op = price_item.select_one(".prices-and-fees__operation-type")
        price = price_item.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)
        result["price_raw"] = price_item.get_text(separator=" ", strip=True)
    else:
        op = soup.select_one(".prices-and-fees__operation-type")
        price = soup.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)

    # Lokasi
    loc = soup.select_one("#view-map__text") or soup.select_one(".view-map__text")
    result["location"] = _text_or_none(loc)

    # Detail tempat
    place_details = soup.select_one(".place-details")
    if place_details:
        items = place_details.select(".details-item-value")
        for it in items:
            txt = _text_or_none(it)
            if not txt:
                continue
            result["raw_details_list"].append(txt)

            lower = txt.lower()
            if "kamar tidur" in lower or "bedroom" in lower:
                result["bedrooms"] = _extract_number_from_text(lower)
                if result["bedrooms"] is None:
                    result["bedrooms"] = txt
            elif "kamar mandi" in lower or "bath" in lower:
                if "kamar mandi" in lower:
                    result["bathrooms"] = _extract_number_from_text(lower) or txt
                elif "guest toilet" in lower or "guest" in lower:
                    result["guest_toilet"] = txt
                else:
                    result["bathrooms"] = _extract_number_from_text(lower) or txt
            elif "guest" in lower or "toilet" in lower:
                result["guest_toilet"] = txt

    # Fitur
    pf = soup.select_one(".place-features")
    if pf:
        for f in pf.select(".feature"):
            b_tag = f.find("b")
            if b_tag:
                key = b_tag.get_text(separator=" ", strip=True).rstrip(":").strip()
                full = f.get_text(separator=" ", strip=True)
                value = full.replace(b_tag.get_text(), "").strip()
                value = re.sub(r'\s+', ' ', value).strip()
                result["place_features"][key] = value if value != "" else None
            else:
                txt = f.get_text(separator=" ", strip=True)
                if txt:
                    result["place_features"].setdefault("other", []).append(txt)

    # Fasilitas
    facilities = soup.select(".facilities")
    for fac in facilities:
        title_el = fac.select_one(".facilities__title")
        title = _text_or_none(title_el)
        items: List[str] = []
        for span in fac.select("li span"):
            t = _text_or_none(span)
            if t:
                items.append(t)
        if not title:
            result.setdefault("facilities_other", []).extend(items)
        else:
            tl = title.lower()
            if "karakteristik properti" in tl or "properti" in tl:
                result["facilities_property"] = items
            elif "karakteristik bangunan" in tl or "bangunan" in tl:
                result["facilities_building"] = items
            else:
                result.setdefault("facilities_other_named", {})[title] = items

    return result


def scrape_mitula(url: str, use_selenium_if_needed: bool = True) -> Dict[str, Any]:
    # Mencoba requests
    soup = _get_soup_requests(url)
    if soup:
        parsed = parse_listing(soup)
        # Jika salah satu data utama ditemukan, return
        if parsed.get("price") or parsed.get("location") or parsed.get("raw_details_list") or parsed.get("latitude"):
            return parsed

    # Jika gagal, coba selenium
    if use_selenium_if_needed:
        soup2 = _get_soup_selenium(url)
        if soup2:
            return parse_listing(soup2)

    return {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "latitude": None,
        "longitude": None,
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": None,
        "facilities_property": None,
        "facilities_building": None,
        "raw_details_list": None
    }


if __name__ == "__main__":
    # Link contoh
    url = ("https://rumah.mitula.co.id/adform/24301-256-7710-4034d3b270fd-afe5-198b749-f7b1?page=1&pos=11&t_sec=1&t_pvid=ad6fb512-d18c-4204-866e-ac472bc3c7ff&hd=false")
    data = scrape_mitula(url, use_selenium_if_needed=True)
    print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "operation_type": null,
  "price": "Rp 40Jt",
  "price_raw": "Rp 40Jt",
  "location": "Kebonagung, Sukodono, Sidoarjo, Jawa Timur",
  "latitude": "-7.4278925",
  "longitude": "112.6813102",
  "bedrooms": 2,
  "bathrooms": 1,
  "guest_toilet": null,
  "place_features": {
    "Jenis rumah": "Rumah",
    "Jenis transaksi": "Beli",
    "Jumlah lantai": "1",
    "Land area": "120 m²"
  },
  "facilities_property": [
    "Garasi",
    "Dapur lengkap",
    "Halaman",
    "Internet",
    "Kabel video",
    "Tanpa perabotan"
  ],
  "facilities_building": [
    "Keamanan",
    "Taman",
    "Area anak-anak",
    "Lapangan tenis"
  ],
  "raw_details_list": [
    "2 kamar tidur",
    "1 kamar mandi",
    "45 m²"
  ]
}


In [ ]:
# scrape_mitula_batch.py
import re
import json
import time
import random
from typing import Optional, Dict, Any, List
import pandas as pd
from tqdm import tqdm

import requests
from bs4 import BeautifulSoup

# Optional selenium imports (only used if you set use_selenium=True)
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except Exception:
    SELENIUM_AVAILABLE = False

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Bot/0.1; +https://example.com/bot)"
}


def _get_soup_requests(url: str, timeout: int = 15) -> Optional[BeautifulSoup]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "lxml")
    except Exception:
        return None

def _get_soup_selenium(url: str, timeout: int = 20) -> Optional[BeautifulSoup]:
    if not SELENIUM_AVAILABLE:
        return None
    driver = None
    try:
        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)
        driver.set_page_load_timeout(timeout)
        driver.get(url)
        html = driver.page_source
        driver.quit()
        return BeautifulSoup(html, "lxml")
    except Exception:
        try:
            if driver:
                driver.quit()
        except Exception:
            pass
        return None


def _text_or_none(el) -> Optional[str]:
    if el is None:
        return None
    txt = el.get_text(separator=" ", strip=True)
    return txt if txt != "" else None


def _extract_number_from_text(text: str) -> Optional[int]:
    if not text:
        return None
    m = re.search(r'(\d+)', text.replace(".", ""))
    if m:
        try:
            return int(m.group(1))
        except Exception:
            return None
    return None


def parse_listing(soup: BeautifulSoup) -> Dict[str, Any]:
    result: Dict[str, Any] = {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": {},
        "facilities_property": [],
        "facilities_building": [],
        "raw_details_list": []
    }

    # Price & operation
    price_item = soup.select_one(".prices-and-fees__price-item.selected")
    if price_item:
        op = price_item.select_one(".prices-and-fees__operation-type")
        price = price_item.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)
        result["price_raw"] = price_item.get_text(separator=" ", strip=True)
    else:
        op = soup.select_one(".prices-and-fees__operation-type")
        price = soup.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)

    # Location
    loc = soup.select_one("#view-map__text") or soup.select_one(".view-map__text")
    result["location"] = _text_or_none(loc)

    # Place details
    place_details = soup.select_one(".place-details")
    if place_details:
        items = place_details.select(".details-item-value")
        for it in items:
            txt = _text_or_none(it)
            if not txt:
                continue
            result["raw_details_list"].append(txt)
            lower = txt.lower()
            if "kamar tidur" in lower or "bedroom" in lower:
                result["bedrooms"] = _extract_number_from_text(lower) or txt
            elif "kamar mandi" in lower:
                result["bathrooms"] = _extract_number_from_text(lower) or txt
            elif "guest" in lower or "toilet" in lower:
                # guest toilet
                result["guest_toilet"] = txt

    # Place features
    pf = soup.select_one(".place-features")
    if pf:
        for f in pf.select(".feature"):
            b_tag = f.find("b")
            if b_tag:
                key = b_tag.get_text(separator=" ", strip=True).rstrip(":").strip()
                full = f.get_text(separator=" ", strip=True)
                value = full.replace(b_tag.get_text(), "").strip()
                value = re.sub(r'\s+', ' ', value).strip()
                result["place_features"][key] = value if value != "" else None
            else:
                txt = f.get_text(separator=" ", strip=True)
                if txt:
                    result["place_features"].setdefault("other", []).append(txt)

    # Facilities (possibly multiple groups)
    facilities = soup.select(".facilities")
    for fac in facilities:
        title_el = fac.select_one(".facilities__title")
        title = _text_or_none(title_el)
        items: List[str] = []
        for span in fac.select("li span"):
            t = _text_or_none(span)
            if t:
                items.append(t)
        if title and ("properti" in title.lower() or "karakteristik properti" in title.lower()):
            result["facilities_property"].extend(items)
        elif title and ("bangunan" in title.lower() or "karakteristik bangunan" in title.lower()):
            result["facilities_building"].extend(items)
        else:
            # unknown group, put under property if property empty; else append to building
            if not result["facilities_property"]:
                result["facilities_property"].extend(items)
            else:
                result["facilities_building"].extend(items)

    # Normalize empty -> None as requested
    if not result["raw_details_list"]:
        result["raw_details_list"] = None
    if not result["place_features"]:
        result["place_features"] = None
    if not result["facilities_property"]:
        result["facilities_property"] = None
    if not result["facilities_building"]:
        result["facilities_building"] = None

    return result


def scrape_mitula(url: str, use_selenium_if_needed: bool = True) -> Dict[str, Any]:
    soup = _get_soup_requests(url)
    if soup:
        parsed = parse_listing(soup)
        if parsed.get("price") or parsed.get("location") or parsed.get("raw_details_list"):
            return parsed
    if use_selenium_if_needed:
        soup2 = _get_soup_selenium(url)
        if soup2:
            return parse_listing(soup2)
    # fallback all None
    return {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": None,
        "facilities_property": None,
        "facilities_building": None,
        "raw_details_list": None
    }


def main(csv_path: str,
         url_column: str = "full_url",
         output_jsonl: str = "mitula_output.jsonl",
         output_csv: str = "mitula_output.csv",
         use_selenium: bool = False,
         delay_min: float = 1.0,
         delay_max: float = 2.5):
    df = pd.read_csv(csv_path)
    if url_column not in df.columns:
        raise ValueError(f"CSV tidak memiliki kolom '{url_column}'")

    results = []
    with open(output_jsonl, "w", encoding="utf-8") as fout:
        for idx, url in enumerate(tqdm(df[url_column].dropna(), desc="Scraping URLs")):
            try:
                item = scrape_mitula(url, use_selenium_if_needed=use_selenium)
                # include source URL and index
                item_record = {"source_url": url, "index": int(idx), **item}
                results.append(item_record)
                fout.write(json.dumps(item_record, ensure_ascii=False) + "\n")
            except Exception as e:
                # record error row with None values
                err_record = {"source_url": url, "index": int(idx), "error": str(e)}
                results.append(err_record)
                fout.write(json.dumps(err_record, ensure_ascii=False) + "\n")

            # polite crawling delay
            time.sleep(random.uniform(delay_min, delay_max))

    # Also save a flattened CSV (NaN will be used for None)
    df_out = pd.json_normalize(results)
    df_out.to_csv(output_csv, index=False)
    print(f"Saved JSONL -> {output_jsonl}")
    print(f"Saved CSV   -> {output_csv}")


if __name__ == "__main__":
    # contoh pemakaian: ubah path CSV bila perlu
    main(
        csv_path="combined_mitula_links.csv",
        url_column="full_url",
        output_jsonl="mitula_output.jsonl",
        output_csv="mitula_output.csv",
        use_selenium=False,
        delay_min=1.0,
        delay_max=2.0
    )


In [4]:
# scrape_mitula_batch.py
import re
import json
import time
import random
from typing import Optional, Dict, Any, List
import pandas as pd
from tqdm import tqdm

import requests
from bs4 import BeautifulSoup

# Optional selenium imports (only used if you set use_selenium=True)
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except Exception:
    SELENIUM_AVAILABLE = False

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Bot/0.1; +https://example.com/bot)"
}


def _get_soup_requests(url: str, timeout: int = 15) -> Optional[BeautifulSoup]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, "lxml")
    except Exception:
        return None

def _get_soup_selenium(url: str, timeout: int = 20) -> Optional[BeautifulSoup]:
    if not SELENIUM_AVAILABLE:
        return None
    driver = None
    try:
        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)
        driver.set_page_load_timeout(timeout)
        driver.get(url)
        html = driver.page_source
        driver.quit()
        return BeautifulSoup(html, "lxml")
    except Exception:
        try:
            if driver:
                driver.quit()
        except Exception:
            pass
        return None


def _text_or_none(el) -> Optional[str]:
    if el is None:
        return None
    txt = el.get_text(separator=" ", strip=True)
    return txt if txt != "" else None


def _extract_number_from_text(text: str) -> Optional[int]:
    if not text:
        return None
    m = re.search(r'(\d+)', text.replace(".", ""))
    if m:
        try:
            return int(m.group(1))
        except Exception:
            return None
    return None


def parse_listing(soup: BeautifulSoup) -> Dict[str, Any]:
    result: Dict[str, Any] = {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "latitude": None,   # [BARU] Field untuk Latitude
        "longitude": None,  # [BARU] Field untuk Longitude
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": {},
        "facilities_property": [],
        "facilities_building": [],
        "raw_details_list": []
    }

    # --- [MODIFIKASI] Ekstraksi Latitude & Longitude via Regex ---
    # Kita ubah soup menjadi string untuk pencarian pola teks JS
    html_content = str(soup)

    # Mencari pola: latitude: "-7.xxx" atau latitude: '-7.xxx'
    # Pola regex menangkap nilai di dalam tanda kutip
    lat_match = re.search(r'latitude\s*:\s*["\']([^"\']+)["\']', html_content)
    lon_match = re.search(r'longitude\s*:\s*["\']([^"\']+)["\']', html_content)

    if lat_match:
        result["latitude"] = lat_match.group(1)
    if lon_match:
        result["longitude"] = lon_match.group(1)
    # -------------------------------------------------------------

    # Price & operation
    price_item = soup.select_one(".prices-and-fees__price-item.selected")
    if price_item:
        op = price_item.select_one(".prices-and-fees__operation-type")
        price = price_item.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)
        result["price_raw"] = price_item.get_text(separator=" ", strip=True)
    else:
        op = soup.select_one(".prices-and-fees__operation-type")
        price = soup.select_one(".prices-and-fees__price")
        result["operation_type"] = _text_or_none(op)
        result["price"] = _text_or_none(price)

    # Location
    loc = soup.select_one("#view-map__text") or soup.select_one(".view-map__text")
    result["location"] = _text_or_none(loc)

    # Place details
    place_details = soup.select_one(".place-details")
    if place_details:
        items = place_details.select(".details-item-value")
        for it in items:
            txt = _text_or_none(it)
            if not txt:
                continue
            result["raw_details_list"].append(txt)
            lower = txt.lower()
            if "kamar tidur" in lower or "bedroom" in lower:
                result["bedrooms"] = _extract_number_from_text(lower) or txt
            elif "kamar mandi" in lower:
                result["bathrooms"] = _extract_number_from_text(lower) or txt
            elif "guest" in lower or "toilet" in lower:
                # guest toilet
                result["guest_toilet"] = txt

    # Place features
    pf = soup.select_one(".place-features")
    if pf:
        for f in pf.select(".feature"):
            b_tag = f.find("b")
            if b_tag:
                key = b_tag.get_text(separator=" ", strip=True).rstrip(":").strip()
                full = f.get_text(separator=" ", strip=True)
                value = full.replace(b_tag.get_text(), "").strip()
                value = re.sub(r'\s+', ' ', value).strip()
                result["place_features"][key] = value if value != "" else None
            else:
                txt = f.get_text(separator=" ", strip=True)
                if txt:
                    result["place_features"].setdefault("other", []).append(txt)

    # Facilities (possibly multiple groups)
    facilities = soup.select(".facilities")
    for fac in facilities:
        title_el = fac.select_one(".facilities__title")
        title = _text_or_none(title_el)
        items: List[str] = []
        for span in fac.select("li span"):
            t = _text_or_none(span)
            if t:
                items.append(t)
        if title and ("properti" in title.lower() or "karakteristik properti" in title.lower()):
            result["facilities_property"].extend(items)
        elif title and ("bangunan" in title.lower() or "karakteristik bangunan" in title.lower()):
            result["facilities_building"].extend(items)
        else:
            # unknown group, put under property if property empty; else append to building
            if not result["facilities_property"]:
                result["facilities_property"].extend(items)
            else:
                result["facilities_building"].extend(items)

    # Normalize empty -> None as requested
    if not result["raw_details_list"]:
        result["raw_details_list"] = None
    if not result["place_features"]:
        result["place_features"] = None
    if not result["facilities_property"]:
        result["facilities_property"] = None
    if not result["facilities_building"]:
        result["facilities_building"] = None

    return result


def scrape_mitula(url: str, use_selenium_if_needed: bool = True) -> Dict[str, Any]:
    soup = _get_soup_requests(url)
    if soup:
        parsed = parse_listing(soup)
        # Update validasi: jika dapat lat/long juga dianggap sukses
        if parsed.get("price") or parsed.get("location") or parsed.get("raw_details_list") or parsed.get("latitude"):
            return parsed

    if use_selenium_if_needed:
        soup2 = _get_soup_selenium(url)
        if soup2:
            return parse_listing(soup2)

    # fallback all None (Update struktur fallback)
    return {
        "operation_type": None,
        "price": None,
        "price_raw": None,
        "location": None,
        "latitude": None,   # [BARU]
        "longitude": None,  # [BARU]
        "bedrooms": None,
        "bathrooms": None,
        "guest_toilet": None,
        "place_features": None,
        "facilities_property": None,
        "facilities_building": None,
        "raw_details_list": None
    }


def main(csv_path: str,
         url_column: str = "full_url",
         output_jsonl: str = "mitula_output.jsonl",
         output_csv: str = "mitula_output.csv",
         use_selenium: bool = False,
         delay_min: float = 1.0,
         delay_max: float = 2.5):

    df = pd.read_csv(csv_path)
    if url_column not in df.columns:
        raise ValueError(f"CSV tidak memiliki kolom '{url_column}'")

    results = []
    # Membuka file JSONL untuk streaming write (aman jika script putus di tengah)
    with open(output_jsonl, "w", encoding="utf-8") as fout:
        for idx, url in enumerate(tqdm(df[url_column].dropna(), desc="Scraping URLs")):
            try:
                item = scrape_mitula(url, use_selenium_if_needed=use_selenium)
                # include source URL and index
                item_record = {"source_url": url, "index": int(idx), **item}
                results.append(item_record)
                fout.write(json.dumps(item_record, ensure_ascii=False) + "\n")
            except Exception as e:
                # record error row with None values
                err_record = {"source_url": url, "index": int(idx), "error": str(e)}
                results.append(err_record)
                fout.write(json.dumps(err_record, ensure_ascii=False) + "\n")

            # polite crawling delay
            time.sleep(random.uniform(delay_min, delay_max))

    # Also save a flattened CSV (NaN will be used for None)
    if results:
        df_out = pd.json_normalize(results)
        df_out.to_csv(output_csv, index=False)
        print(f"Saved JSONL -> {output_jsonl}")
        print(f"Saved CSV    -> {output_csv}")
    else:
        print("Tidak ada data yang berhasil diambil.")


if __name__ == "__main__":
    # contoh pemakaian: ubah path CSV bila perlu
    main(
        csv_path="/content/new_combined_mitula_links.csv", # Ganti dengan nama file CSV input Anda
        url_column="full_url",                # Pastikan nama kolom link di CSV benar
        output_jsonl="mitula_output_latlong.jsonl",
        output_csv="mitula_output_latlong.csv",
        use_selenium=False,                   # Set True jika request biasa banyak gagal
        delay_min=1.0,
        delay_max=2.0
    )

Scraping URLs: 100%|██████████| 4200/4200 [3:18:09<00:00,  2.83s/it]

Saved JSONL -> mitula_output_latlong.jsonl
Saved CSV    -> mitula_output_latlong.csv
